# Pipeline-uri ML si Prevenirea Data Leakage - Tema

Bine ai venit la tema despre Pipeline-uri ML si Prevenirea Data Leakage.

Pipeline-urile de machine learning sunt instrumente esentiale pentru construirea unor modele robuste, reproductibile si pregatite pentru productie. Un pipeline leaga intre ele mai multi pasi de procesare, de la preprocesarea datelor pana la antrenarea modelului, asigurand ca operatiile sunt aplicate consecvent atat la antrenare, cat si la inferenta. Pipeline-urile proiectate corect previn data leakage, simplifica optimizarea hiperparametrilor si fac modelele mai usor de deploy-at si intretinut.

Data leakage apare atunci cand informatii din afara setului de antrenare sunt folosite pentru a crea modelul, ceea ce duce la estimari excesiv de optimiste ale performantei, care esueaza in productie. Leakage-ul poate fi subtil: fit pe scaler pe intregul set de date inainte de split, folosirea unor caracteristici dependente de tinta sau aplicarea unor transformari care "vad" viitorul. Intelegerea si prevenirea leakage-ului sunt esentiale pentru construirea unor modele care generalizeaza in lumea reala.

In aceasta tema vei invata sa construiesti pipeline-uri care mentin limite corecte intre date, gestioneaza tipuri eterogene de date, se integreaza cu hyperparameter tuning si pregatesc modelele pentru deployment in productie. De asemenea, vei identifica tipare comune de leakage si vei implementa masuri de protectie impotriva lor.

**Ce vei face in aceasta tema**

* Construiesti pipeline-uri de baza care leaga corect preprocesarea si modelarea.
* Folosesti ColumnTransformer pentru a gestiona unitar date numerice si categorice.
* Identifici si corectezi scenarii de data leakage din exemplele de cod oferite.
* Integrezi pipeline-uri cu GridSearchCV pentru hyperparameter tuning sigur.
* Creezi pipeline-uri pregatite pentru productie, care gestioneaza date brute end-to-end.
* Implementezi logica unui pipeline de la zero pentru a intelege mecanica fit/transform.

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul solutiei. **Nu adauga si nu modifica niciun cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar evaluatorul le va ignora, asa ca nu te baza pe celulele create de tine pentru codul de raspuns; foloseste spatiile oferite in notebook.

* Evita folosirea variabilelor globale, cu exceptia situatiilor absolut necesare. Evaluatorul iti testeaza codul intr-un mediu izolat fara sa ruleze toate celulele de la inceput. Din acest motiv, variabilele globale pot sa nu fie disponibile in timpul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai din iconita de salvare din stanga sus a paginii, apoi apasa pe butonul `Submit assignment` din dreapta sus.
---


## Cuprins

1. [Introducere in pipeline-urile ML](#1)
2. [Construirea unui pipeline de baza](#2)
   - [Exercitiul 1: Construieste un pipeline de baza](#ex-1)
3. [ColumnTransformer pentru date mixte](#3)
   - [Exercitiul 2: Implementeaza ColumnTransformer](#ex-2)
4. [Identificarea data leakage](#4)
   - [Exercitiul 3: Gaseste si corecteaza leakage-ul](#ex-3)
5. [Pipeline cu hyperparameter tuning](#5)
   - [Exercitiul 4: Pipeline cu GridSearchCV](#ex-4)
6. [Pipeline de productie](#6)
   - [Exercitiul 5: Pipeline de productie end-to-end](#ex-5)
7. [Implementare de la zero](#7)
   - [Exercitiul 6: Pipeline simplu de la zero](#ex-6)
8. [Concluzie](#8)


<a name='1'></a>
## 1 - Introducere in pipeline-urile ML

**Pipeline-urile** automatizeaza fluxul de transformare a datelor si antrenare a modelelor, asigurand consistenta si prevenind erori comune.

### De ce sa folosesti pipeline-uri?

1. **Previn data leakage**: se asigura ca preprocesarea face fit doar pe datele de antrenare
2. **Reproductibilitate**: aceleasi transformari sunt aplicate in mod consistent
3. **Simplifica codul**: flux clar si usor de citit
4. **Permit Grid Search**: tuning comun pentru preprocesare si hiperparametri ai modelului
5. **Pregatite pentru productie**: serializare si deployment usoare

### Structura unui pipeline:

Un pipeline este o secventa de pasi:
$$\text{Pipeline} = [\text{Step}_1, \text{Step}_2, ..., \text{Step}_n, \text{Estimator}]$$

Fiecare pas (cu exceptia ultimului) trebuie sa implementeze:
- `fit(X, y)`: invata parametrii din datele de antrenare
- `transform(X)`: aplica transformarea pe date

Pasul final (estimatorul) implementeaza:
- `fit(X, y)`: antreneaza modelul
- `predict(X)`: face predictii

### Executia pipeline-ului:

**Training** (`pipeline.fit(X_train, y_train)`):
```python
# Step 1: Fit and transform
X_temp = step1.fit(X_train, y_train).transform(X_train)
# Step 2: Fit and transform
X_temp = step2.fit(X_temp, y_train).transform(X_temp)
# Final: Fit estimator
estimator.fit(X_temp, y_train)
```

**Prediction** (`pipeline.predict(X_test)`):
```python
# Only transform (no fitting!)
X_temp = step1.transform(X_test)
X_temp = step2.transform(X_temp)
predictions = estimator.predict(X_temp)
```

### Data leakage:

**Data leakage** apare atunci cand informatii din test influenteaza antrenarea.

**Surse comune:**
1. **Preprocessing Leakage**: fit pe scaler pe intregul set de date
2. **Temporal Leakage**: folosirea viitorului pentru a prezice trecutul
3. **Target Leakage**: caracteristici derivate din tinta
4. **Test Set Leakage**: folosirea test set-ului pentru orice decizie de antrenare

**Exemplu de leakage:**
```python
# WRONG: Fits scaler on ALL data including test
scaler = StandardScaler().fit(X)  # X includes test data!
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled)

# CORRECT: Fit only on training data
X_train, X_test = train_test_split(X)
scaler = StandardScaler().fit(X_train)  # Only training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

### Principii cheie:

- **Fit doar pe datele de antrenare**: nu face niciodata fit pe transformere cu test / validare
- **Transforma consecvent**: foloseste aceleasi transformere antrenate pentru toate seturile
- **Respecta ordinea temporala**: nu folosi viitorul pentru a prezice trecutul
- **Valideaza atent**: foloseste cross-validation corect, cu pipeline-uri


### Importa bibliotecile necesare


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import unittests
import warnings
warnings.filterwarnings('ignore')

# Pipeline and preprocessing
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

# Models and evaluation
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Datasets
from sklearn.datasets import load_iris, load_breast_cancer, make_classification

# Serialization
import joblib
import pickle

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

### Functii ajutatoare


In [ ]:
def load_titanic_data():
    """Load Titanic dataset for mixed data type examples."""
    try:
        titanic = sns.load_dataset('titanic')
        # Select relevant columns
        titanic = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
        titanic = titanic.dropna(subset=['embarked'])  # Drop few missing embarked values
        
        X = titanic.drop('survived', axis=1)
        y = titanic['survived']
        
        return X, y
    except:
        print("Titanic dataset not available, creating synthetic mixed data")
        # Create synthetic data with mixed types
        n_samples = 800
        df = pd.DataFrame({
            'numeric1': np.random.randn(n_samples),
            'numeric2': np.random.rand(n_samples) * 100,
            'category1': np.random.choice(['A', 'B', 'C'], n_samples),
            'category2': np.random.choice(['X', 'Y'], n_samples)
        })
        y = (df['numeric1'] + df['numeric2'] / 50 + 
             (df['category1'] == 'A').astype(int) * 2 + 
             np.random.randn(n_samples) * 0.5 > 1).astype(int)
        return df, y


def demonstrate_leakage():
    """Create dataset that clearly shows impact of data leakage."""
    np.random.seed(42)
    n_samples = 1000
    
    # Create feature with large variance differences between train/test
    X_train_part = np.random.randn(800, 1) * 10  # High variance
    X_test_part = np.random.randn(200, 1) * 2    # Low variance
    X = np.vstack([X_train_part, X_test_part])
    
    # Target depends on scaled feature
    y = (X[:, 0] > 0).astype(int)
    
    return X, y


def compare_with_without_leakage(X, y):
    """Compare model performance with and without data leakage."""
    # Split data first
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # WITH LEAKAGE: Fit scaler on all data
    scaler_leakage = StandardScaler().fit(X)  # WRONG!
    X_train_leaked = scaler_leakage.transform(X_train)
    X_test_leaked = scaler_leakage.transform(X_test)
    
    model_leaked = LogisticRegression(random_state=42)
    model_leaked.fit(X_train_leaked, y_train)
    score_leaked = model_leaked.score(X_test_leaked, y_test)
    
    # WITHOUT LEAKAGE: Fit scaler only on training data
    scaler_correct = StandardScaler().fit(X_train)  # CORRECT!
    X_train_correct = scaler_correct.transform(X_train)
    X_test_correct = scaler_correct.transform(X_test)
    
    model_correct = LogisticRegression(random_state=42)
    model_correct.fit(X_train_correct, y_train)
    score_correct = model_correct.score(X_test_correct, y_test)
    
    return score_leaked, score_correct


def visualize_pipeline(pipeline, save_path=None):
    """Visualize pipeline structure."""
    from sklearn import set_config
    set_config(display='diagram')
    return pipeline


def evaluate_pipeline(pipeline, X_train, X_test, y_train, y_test):
    """Evaluate pipeline performance."""
    # Train
    start = time()
    pipeline.fit(X_train, y_train)
    train_time = time() - start
    
    # Evaluate
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    
    # Predictions
    y_pred = pipeline.predict(X_test)
    
    results = {
        'train_score': train_score,
        'test_score': test_score,
        'train_time': train_time,
        'predictions': y_pred
    }
    
    return results

print("Helper functions defined!")

<a name='2'></a>
## 2 - Construirea unui pipeline de baza

Un pipeline de baza leaga pasii de preprocesare de model, asigurand ordinea corecta fit/transform.

### Sintaxa pipeline-ului:

**Metoda 1: Pipeline cu pasi numiti**
```python
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
```

**Metoda 2: make_pipeline (nume generate automat)**
```python
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
```

### Operatii ale pipeline-ului:

**Training:**
```python
pipeline.fit(X_train, y_train)  # Fits all steps on training data
```

**Prediction:**
```python
predictions = pipeline.predict(X_test)  # Transforms and predicts
```

**Accesarea pasilor:**
```python
pipeline.named_steps['scaler']  # Access specific step
pipeline.steps[0][1]            # Access by index
pipeline['scaler']              # Dictionary-like access
```

### Beneficii:

1. **Previne leakage-ul**: scaler-ul face fit doar pe train
2. **Cod mai curat**: un singur obiect in loc de mai multe
3. **Cross-Validation**: functioneaza natural cu CV
4. **Serializare**: salvezi intregul pipeline ca un singur obiect

### Pasi uzuali intr-un pipeline:

- **Imputare**: `SimpleImputer()` pentru valori lipsa
- **Scalare**: `StandardScaler()`, `MinMaxScaler()`, `RobustScaler()`
- **Selectie de caracteristici**: `SelectKBest()`, `PCA()`
- **Encoding**: `OneHotEncoder()`, `OrdinalEncoder()`
- **Model**: orice clasificator sau regressor


<a name='ex-1'></a>
### Exercitiul 1 - Construieste un pipeline de baza

**Sarcina ta:**

Implementeaza functia `create_basic_pipeline` astfel incat:
1. Sa creeze un pipeline cu StandardScaler si un clasificator
2. Sa antreneze pipeline-ul pe datele de antrenare
3. Sa evalueze pe datele de test
4. Sa compare cu abordarea manuala (fara pipeline)

<details>
  <summary><b><font color="green">Indicii de cod (apasati pentru extindere daca ai nevoie de ajutor)</font></b></summary>
  
**Pentru crearea pipeline-ului:**
* Foloseste Pipeline: `pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', model)])`
* Sau make_pipeline: `pipeline = make_pipeline(StandardScaler(), model)`

**Pentru antrenare:**
* Apel unic: `pipeline.fit(X_train, y_train)`
* Gestioneaza intern fit si transform

**Pentru evaluare:**
* Score: `pipeline.score(X_test, y_test)`
* Predict: `pipeline.predict(X_test)`

**Pentru comparatie:**
* Manual: fit pe scaler, transform train, transform test, fit pe model, predict
* Pipeline: un singur apel de fit si predict

</details>


In [ ]:
# GRADED FUNCTION: create_basic_pipeline
def create_basic_pipeline(X_train, X_test, y_train, y_test, model=None):
    """
    Create and evaluate a basic preprocessing + model pipeline.
    
    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        model: Classifier to use (default: LogisticRegression)
    
    Returns:
        Dictionary with pipeline, train_score, test_score
    """
    if model is None:
        model = LogisticRegression(random_state=42, max_iter=1000)
    
    ### ÎNCEPUT DE COD AICI ### (≈ 10 lines)
    
    # Create pipeline with scaler and classifier
    pipeline = None
    
    # Fit pipeline on training data
    
    # Evaluate on training data
    train_score = None
    
    # Evaluate on test data
    test_score = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'pipeline': pipeline,
        'train_score': train_score,
        'test_score': test_score
    }

In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

print("Dataset loaded:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")

# Create and evaluate basic pipeline
print("\n" + "="*60)
print("BASIC PIPELINE")
print("="*60)

results = create_basic_pipeline(X_train, X_test, y_train, y_test)

print(f"\nPipeline created successfully!")
print(f"Training accuracy: {results['train_score']:.4f}")
print(f"Test accuracy: {results['test_score']:.4f}")

# Display pipeline structure
print(f"\nPipeline steps:")
for name, step in results['pipeline'].named_steps.items():
    print(f"  {name}: {step.__class__.__name__}")

# Compare with manual approach
print("\n" + "="*60)
print("COMPARISON: PIPELINE vs MANUAL")
print("="*60)

# Manual approach (more verbose, error-prone)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)
manual_score = model.score(X_test_scaled, y_test)

print(f"\nManual approach test score: {manual_score:.4f}")
print(f"Pipeline test score: {results['test_score']:.4f}")
print(f"Difference: {abs(manual_score - results['test_score']):.6f} (should be ~0)")

print("\nBenefits of Pipeline:")
print("  • Prevents accidentally fitting on test data")
print("  • Cleaner, more readable code")
print("  • Easier to serialize and deploy")
print("  • Works seamlessly with cross-validation")

##### **Rezultatul asteptat**
```
Dataset loaded:
Training set: (105, 4)
Test set: (45, 4)
Number of classes: 3

============================================================
BASIC PIPELINE
============================================================

Pipeline created successfully!
Training accuracy: 0.9XXX
Test accuracy: 0.9XXX

Pipeline steps:
  scaler: StandardScaler
  classifier: LogisticRegression

============================================================
COMPARATIE: PIPELINE vs MANUAL
============================================================

Manual approach test score: 0.9XXX
Pipeline test score: 0.9XXX
Difference: 0.000000 (should be ~0)
```
Abordarea cu pipeline si cea manuala ar trebui sa produca rezultate identice, dar pipeline-ul este mai curat si mai sigur.


In [ ]:
unittests.exercise_1(create_basic_pipeline)

<a name='3'></a>
## 3 - ColumnTransformer pentru date mixte

Seturile de date din lumea reala contin adesea atat caracteristici numerice, cat si categorice. **ColumnTransformer** aplica preprocesari diferite pe tipuri diferite de coloane.

### Provocarea:

Avem un set de date cu:
- **coloane numerice**: `age`, `fare` -> necesita scalare
- **coloane categorice**: `sex`, `embarked` -> necesita encoding

Avem nevoie de transformari diferite pentru fiecare tip!

### Sintaxa ColumnTransformer:

```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])
```

### Pipeline complet:

```python
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])
```

### Fluxul de executie:

1. **ColumnTransformer** aplica:
   - `StandardScaler()` pe coloanele numerice
   - `OneHotEncoder()` pe coloanele categorice
   - concateneaza rezultatele

2. **Classifier** primeste caracteristicile concatenate

### Optiuni avansate:

**Gestionarea remainder:**
```python
ColumnTransformer(
    transformers=[...],
    remainder='passthrough'  # Keep untransformed columns
    # remainder='drop'       # Drop untransformed columns (default)
)
```

**make_column_transformer (nume generate automat):**
```python
from sklearn.compose import make_column_transformer

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(), categorical_features)
)
```

### Tipare comune:

**Preprocesare numerica:**
- imputare: `SimpleImputer(strategy='median')`
- scalare: `StandardScaler()`, `MinMaxScaler()`, `RobustScaler()`

**Preprocesare categorica:**
- imputare: `SimpleImputer(strategy='most_frequent')`
- encoding: `OneHotEncoder(handle_unknown='ignore')`

**Combinarea ambelor:**
```python
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
```


<a name='ex-2'></a>
### Exercitiul 2 - Implementeaza ColumnTransformer

**Sarcina ta:**

Implementeaza functia `create_column_transformer_pipeline` astfel incat:
1. Sa identifice coloanele numerice si categorice
2. Sa creeze transformere separate pentru fiecare tip
3. Sa le combine cu ColumnTransformer
4. Sa construiasca un pipeline complet cu un clasificator

<details>
  <summary><b><font color="green">Indicii de cod (apasati pentru extindere daca ai nevoie de ajutor)</font></b></summary>
  
**Pentru identificarea coloanelor:**
* Numeric: `X.select_dtypes(include=['int64', 'float64']).columns.tolist()`
* Categorical: `X.select_dtypes(include=['object', 'category']).columns.tolist()`

**Pentru transformere:**
* Pipeline numeric: Imputer -> Scaler
* Pipeline categoric: Imputer -> Encoder
* Foloseste `handle_unknown='ignore'` in OneHotEncoder

**Pentru ColumnTransformer:**
```python
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
```

**Pentru pipeline-ul complet:**
* Combina preprocessor cu clasificatorul

</details>


In [ ]:
# GRADED FUNCTION: create_column_transformer_pipeline
def create_column_transformer_pipeline(X_train, y_train, X_test, y_test, model=None):
    """
    Create pipeline with ColumnTransformer for mixed data types.
    
    Args:
        X_train: Training features (DataFrame with mixed types)
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        model: Classifier to use (default: RandomForestClassifier)
    
    Returns:
        Dictionary with pipeline, train_score, test_score, feature_names
    """
    if model is None:
        model = RandomForestClassifier(random_state=42, n_estimators=100)
    
    ### ÎNCEPUT DE COD AICI ### (≈ 25 lines)
    
    # Identify numeric and categorical columns
    numeric_features = None
    categorical_features = None
    
    # Create numeric transformer (imputation + scaling)
    numeric_transformer = None
    
    # Create categorical transformer (imputation + encoding)
    categorical_transformer = None
    
    # Combine transformers with ColumnTransformer
    preprocessor = None
    
    # Create full pipeline
    pipeline = None
    
    # Fit pipeline
    # pipeline.fit(...)
    
    # Evaluate
    train_score = None
    test_score = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'pipeline': pipeline,
        'train_score': train_score,
        'test_score': test_score,
        'numeric_features': numeric_features,
        'categorical_features': categorical_features
    }

In [ ]:
# Load Titanic dataset (mixed numeric/categorical)
X_titanic, y_titanic = load_titanic_data()

print("Dataset loaded:")
print(f"Shape: {X_titanic.shape}")
print(f"\nFeatures and types:")
print(X_titanic.dtypes)
print(f"\nFirst few rows:")
print(X_titanic.head())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42
)

print("\n" + "="*60)
print("COLUMNTRANSFORMER PIPELINE")
print("="*60)

# Create and evaluate pipeline
results = create_column_transformer_pipeline(X_train, y_train, X_test, y_test)

print(f"\nPipeline created successfully!")
print(f"\nNumeric features ({len(results['numeric_features'])}): {results['numeric_features']}")
print(f"Categorical features ({len(results['categorical_features'])}): {results['categorical_features']}")
print(f"\nTraining accuracy: {results['train_score']:.4f}")
print(f"Test accuracy: {results['test_score']:.4f}")

# Display pipeline structure
print(f"\nPipeline structure:")
print(results['pipeline'])

# Make sample prediction
sample = X_test.iloc[:1]
prediction = results['pipeline'].predict(sample)
print(f"\nSample prediction:")
print(f"Input: {sample.to_dict('records')[0]}")
print(f"Prediction: {'Survived' if prediction[0] == 1 else 'Did not survive'}")

##### **Rezultatul asteptat**
```
Dataset loaded:
Shape: (XXX, X)

Features and types:
pclass        int64
sex          object
age         float64
...

============================================================
COLUMNTRANSFORMER PIPELINE
============================================================

Pipeline created successfully!

Numeric features (X): ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical features (X): ['sex', 'embarked']

Training accuracy: 0.9XXX
Test accuracy: 0.7XXX
```
Pipeline-ul ar trebui sa gestioneze corect atat caracteristicile numerice, cat si pe cele categorice, aplicand automat transformarile potrivite.


In [ ]:
unittests.exercise_2(create_column_transformer_pipeline)

<a name='4'></a>
## 4 - Identificarea data leakage

**Data leakage** este una dintre cele mai comune si mai periculoase greseli in machine learning. Ea duce la modele care performeaza bine in dezvoltare, dar esueaza in productie.

### Tipuri de leakage:

#### 1. **Preprocessing Leakage**

**Problema:** transformer-ele fac fit pe intregul set de date, inclusiv pe test

```python
# WRONG: Scaler sees test data!
scaler = StandardScaler().fit(X)  # Includes test data
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled)

# CORRECT: Fit only on training
X_train, X_test = train_test_split(X)
scaler = StandardScaler().fit(X_train)  # Only training
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

**De ce conteaza:** statisticile din test set (mean, std) influenteaza antrenarea

#### 2. **Target Leakage**

**Problema:** caracteristici care contin informatie despre tinta

```python
# WRONG: 'already_defaulted' is the target in disguise!
features = ['income', 'debt', 'already_defaulted']
target = 'will_default'

# CORRECT: Only use features available at prediction time
features = ['income', 'debt', 'credit_score']
```

**Exemple:**
- prezici o boala folosind teste facute dupa diagnostic
- prezici churn folosind motivul anularii
- prezici actiuni folosind preturi viitoare

#### 3. **Temporal Leakage**

**Problema:** folosesti informatie din viitor pentru a prezice trecutul

```python
# WRONG: Random split on time series
X_train, X_test = train_test_split(timeseries_data)  # Mixes past and future

# CORRECT: Temporal split
split_date = '2023-01-01'
X_train = data[data.date < split_date]
X_test = data[data.date >= split_date]
```

#### 4. **Cross-Validation Leakage**

**Problema:** preprocesare facuta inainte de split-urile CV

```python
# WRONG: Preprocessing outside CV
X_scaled = StandardScaler().fit_transform(X)
scores = cross_val_score(model, X_scaled, y)  # Each fold sees others!

# CORRECT: Use pipeline
pipeline = make_pipeline(StandardScaler(), model)
scores = cross_val_score(pipeline, X, y)  # Proper isolation
```

### Strategii de detectie:

1. **Performanta suspect de mare**: 99%+ accuracy pe probleme complexe
2. **Scadere in productie**: foarte bun in dev, slab in productie
3. **Feature Importance**: caracteristicile similare cu tinta au importanta mare
4. **Verificare temporala**: poti cunoaste aceasta caracteristica inainte de predictie?
5. **Distribution Shift**: distributiile pe train si test difera semnificativ

### Bune practici de prevenire:

- **Foloseste mereu pipeline-uri** pentru preprocesare
- **Fa split mai intai**, apoi preproceseaza
- **Verifica disponibilitatea caracteristicilor** la momentul predictiei
- **Respecta ordinea temporala** in serii de timp
- **Foloseste CV corect** cu pipeline-uri
- **Valideaza pe date cu adevarat hold-out**
- **Documenteaza timestamp-urile** colectarii datelor


<a name='ex-3'></a>
### Exercitiul 3 - Gaseste si corecteaza leakage-ul

**Sarcina ta:**

Implementeaza functia `identify_and_fix_leakage` astfel incat:
1. Sa demonstreze cod cu data leakage
2. Sa arate diferenta de performanta
3. Sa corecteze leakage-ul folosind un pipeline corect
4. Sa explice de ce solutia previne leakage-ul

<details>
  <summary><b><font color="green">Indicii de cod (apasati pentru extindere daca ai nevoie de ajutor)</font></b></summary>
  
**Pentru codul cu leakage:**
* Fa fit pe scaler pe tot X inainte de split
* Apoi fa split pe datele scalate
* Antreneaza modelul pe aceste date

**Pentru codul corect:**
* Fa split-ul MAI INTAI
* Creeaza pipeline cu scaler si model
* Fa fit pe pipeline doar pe datele de antrenare

**Pentru comparatie:**
* Evalueaza ambele variante pe test set
* Versiunea cu leakage poate arata performanta nerealist de mare
* Sau poate avea un comportament neasteptat

**Ideea cheie:**
* Statisticile din test set (mean, std) nu trebuie sa influenteze antrenarea
* Pipeline-ul asigura asta deoarece face fit doar pe fold-ul de antrenare

</details>


In [ ]:
# GRADED FUNCTION: identify_and_fix_leakage
def identify_and_fix_leakage(X, y, test_size=0.2):
    """
    Demonstrate data leakage and how to fix it.
    
    Args:
        X: Features
        y: Target
        test_size: Proportion of test set
    
    Returns:
        Dictionary with leaky_score, correct_score, explanation
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 25 lines)
    
    # LEAKY APPROACH: Fit scaler on ALL data before splitting
    # 1. Fit scaler on entire X
    scaler_leaky = None
    X_scaled_leaky = None
    
    # 2. Split AFTER scaling (WRONG!)
    X_train_leaky, X_test_leaky, y_train, y_test = None
    
    # 3. Train model
    model_leaky = None
    # Fit model
    
    # 4. Evaluate
    leaky_score = None
    
    # CORRECT APPROACH: Use pipeline
    # 1. Split FIRST
    X_train, X_test, y_train, y_test = None
    
    # 2. Create pipeline (fits scaler only on training data)
    pipeline_correct = None
    
    # 3. Fit pipeline
    # pipeline_correct.fit(...)
    
    # 4. Evaluate
    correct_score = None
    
    # Explanation of the fix
    explanation = None  # Write explanation string
    
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'leaky_score': leaky_score,
        'correct_score': correct_score,
        'score_difference': abs(leaky_score - correct_score),
        'explanation': explanation
    }

In [ ]:
# Create dataset that demonstrates leakage impact
X_leak, y_leak = demonstrate_leakage()

print("="*60)
print("DATA LEAKAGE DEMONSTRATION")
print("="*60)

results = identify_and_fix_leakage(X_leak, y_leak)

print(f"\nPerformance Comparison:")
print(f"Leaky approach:   {results['leaky_score']:.4f}")
print(f"Correct approach: {results['correct_score']:.4f}")
print(f"Difference:       {results['score_difference']:.4f}")

print(f"\nExplanation:")
print(results['explanation'])

# Visual demonstration
print("\n" + "="*60)
print("CROSS-VALIDATION WITH AND WITHOUT PIPELINE")
print("="*60)

# Load Iris for CV demonstration
X_iris, y_iris = load_iris(return_X_y=True)

# WRONG: Preprocessing before CV
X_scaled_wrong = StandardScaler().fit_transform(X_iris)
model_wrong = LogisticRegression(random_state=42, max_iter=1000)
scores_wrong = cross_val_score(model_wrong, X_scaled_wrong, y_iris, cv=5)

# CORRECT: Pipeline ensures proper CV
pipeline_correct = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=1000)
)
scores_correct = cross_val_score(pipeline_correct, X_iris, y_iris, cv=5)

print(f"\nCV Scores (preprocessing before CV - WRONG):")
print(f"  Scores: {scores_wrong}")
print(f"  Mean: {scores_wrong.mean():.4f} (+/- {scores_wrong.std():.4f})")

print(f"\nCV Scores (pipeline with proper isolation - CORRECT):")
print(f"  Scores: {scores_correct}")
print(f"  Mean: {scores_correct.mean():.4f} (+/- {scores_correct.std():.4f})")

print("\n⚠️  The 'wrong' approach may show slightly better scores because")
print("    each fold's validation set influenced the scaler during training!")

# Common leakage scenarios
print("\n" + "="*60)
print("COMMON LEAKAGE SCENARIOS")
print("="*60)

leakage_examples = [
    {
        'name': 'Preprocessing Leakage',
        'wrong': 'scaler.fit(X_all); split(X_all)',
        'right': 'split(X); scaler.fit(X_train)',
        'impact': 'Test statistics leak into training'
    },
    {
        'name': 'Target Leakage',
        'wrong': 'Using feature that contains target info',
        'right': 'Only use features available before target',
        'impact': 'Unrealistically high performance'
    },
    {
        'name': 'Temporal Leakage',
        'wrong': 'train_test_split() on time series',
        'right': 'Split by time: train < test chronologically',
        'impact': 'Model sees the future'
    },
    {
        'name': 'CV Leakage',
        'wrong': 'Preprocess before cross_val_score',
        'right': 'Use Pipeline with cross_val_score',
        'impact': 'Folds contaminate each other'
    }
]

for i, ex in enumerate(leakage_examples, 1):
    print(f"\n{i}. {ex['name']}")
    print(f"   Wrong:  {ex['wrong']}")
    print(f"   Right:  {ex['right']}")
    print(f"   Impact: {ex['impact']}")

##### **Rezultatul asteptat**
```
============================================================
DEMONSTRATIE DATA LEAKAGE
============================================================

Comparatia performantei:
Leaky approach:   0.XXXX
Correct approach: 0.XXXX
Difference:       0.XXXX

Explanation:
[Your explanation of why leakage occurred and how pipeline fixes it]

============================================================
CROSS-VALIDATION CU SI FARA PIPELINE
============================================================

CV Scores (preprocessing before CV - WRONG):
  Scores: [0.9XXX 0.9XXX 0.9XXX 0.9XXX 0.9XXX]
  Mean: 0.9XXX (+/- 0.0XXX)

CV Scores (pipeline with proper isolation - CORRECT):
  Scores: [0.9XXX 0.9XXX 0.9XXX 0.9XXX 0.9XXX]
  Mean: 0.9XXX (+/- 0.0XXX)
```
Demonstratia ar trebui sa arate clar cum pipeline-urile previn leakage-ul, asigurand ca preprocesarea face fit doar pe fold-urile de antrenare.


In [ ]:
unittests.exercise_3(identify_and_fix_leakage)

<a name='5'></a>
## 5 - Pipeline cu hyperparameter tuning

Pipeline-urile se integreaza natural cu GridSearchCV si RandomizedSearchCV, permitand tuning sigur pentru preprocesare si hiperparametrii modelului in acelasi timp.

### De ce sa faci tuning in pipeline?

1. **Previne leakage-ul**: preprocesarea este refit pentru fiecare configuratie de hiperparametri
2. **Exploreaza preprocesarea**: poti regla tipul de scaler, strategia de imputare etc.
3. **Optimizare comuna**: gasesti cea mai buna combinatie de preprocesare si parametri ai modelului
4. **Cod mai curat**: totul este intr-un singur obiect

### Denumirea parametrilor:

Accesezi pasii din pipeline folosind double underscore `__`:

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None],
    # Can even tune preprocessing:
    # 'scaler': [StandardScaler(), MinMaxScaler()]
}
```

### GridSearchCV cu pipeline:

```python
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
best_pipeline = grid_search.best_estimator_
```

### Ce se intampla in fiecare fold CV:

Pentru fiecare combinatie de parametri:
1. Split in train / validation
2. **Fit pe scaler doar pe train fold**
3. Transforma train si validation
4. Fit pe model pe train-ul transformat
5. Evalueaza pe validation-ul transformat
6. Repeta pentru toate fold-urile

### Pipeline-uri complexe:

Cu ColumnTransformer:

```python
param_grid = {
    # Numeric preprocessing
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__num__scaler': [StandardScaler(), MinMaxScaler()],
    
    # Categorical preprocessing
    'preprocessor__cat__encoder__handle_unknown': ['ignore', 'error'],
    
    # Model parameters
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10]
}
```

### Bune practici:

- **Incepe mic**: optimizeaza mai intai cativa parametri importanti
- **Foloseste random search**: pentru spatii mari de parametri
- **Seteaza cv adecvat**: echilibru intre runtime si fiabilitate
- **Refit pe modelul final**: GridSearchCV face refit pe toate datele de antrenare cu cei mai buni parametri
- **Extrage best pipeline**: foloseste `best_estimator_` pentru predictii


<a name='ex-4'></a>
### Exercitiul 4 - Pipeline cu GridSearchCV

**Sarcina ta:**

Implementeaza functia `tune_pipeline_hyperparameters` astfel incat:
1. Sa creeze un pipeline cu preprocesare si clasificator
2. Sa defineasca o grila de parametri atat pentru preprocesare, cat si pentru model
3. Sa realizeze grid search cu cross-validation corect
4. Sa returneze best pipeline si rezultatele

<details>
  <summary><b><font color="green">Indicii de cod (apasati pentru extindere daca ai nevoie de ajutor)</font></b></summary>
  
**Pentru crearea pipeline-ului:**
* Include pasii de preprocesare si clasificatorul
* Denumeste clar pasii pentru referentierea parametrilor

**Pentru grila de parametri:**
* Foloseste double underscore: `'step_name__param_name'`
* Exemplu: `'classifier__n_estimators': [50, 100, 200]`
* Poti optimiza si preprocesarea: `'scaler': [StandardScaler(), RobustScaler()]`

**Pentru GridSearchCV:**
```python
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True
)
```

**Pentru rezultate:**
* Best params: `grid_search.best_params_`
* Best score: `grid_search.best_score_`
* Best pipeline: `grid_search.best_estimator_`
* All results: `pd.DataFrame(grid_search.cv_results_)`

</details>


In [ ]:
# GRADED FUNCTION: tune_pipeline_hyperparameters
def tune_pipeline_hyperparameters(X_train, X_test, y_train, y_test, cv_folds=3):
    """
    Tune hyperparameters of a pipeline using GridSearchCV.
    
    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        cv_folds: Number of CV folds
    
    Returns:
        Dictionary with best_pipeline, best_params, best_score, test_score
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 20 lines)
    
    # Create base pipeline
    pipeline = None
    
    # Define parameter grid
    param_grid = None
    
    # Create GridSearchCV
    grid_search = None
    
    # Perform grid search
    # grid_search.fit(...)
    
    # Extract results
    best_pipeline = None
    best_params = None
    best_cv_score = None
    
    # Evaluate on test set
    test_score = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'best_pipeline': best_pipeline,
        'best_params': best_params,
        'best_cv_score': best_cv_score,
        'test_score': test_score,
        'grid_search': grid_search
    }

In [ ]:
# Load dataset
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("="*60)
print("PIPELINE HYPERPARAMETER TUNING")
print("="*60)

results = tune_pipeline_hyperparameters(X_train, X_test, y_train, y_test, cv_folds=3)

print(f"\nGrid Search Completed!")
print(f"\nBest Parameters:")
for param, value in results['best_params'].items():
    print(f"  {param}: {value}")

print(f"\nPerformance:")
print(f"  Best CV Score: {results['best_cv_score']:.4f}")
print(f"  Test Score:    {results['test_score']:.4f}")

# Show top configurations
cv_results = pd.DataFrame(results['grid_search'].cv_results_)
top_results = cv_results.nsmallest(5, 'rank_test_score')[[
    'params', 'mean_test_score', 'std_test_score', 'rank_test_score'
]]

print(f"\nTop 5 Configurations:")
for idx, row in top_results.iterrows():
    print(f"\nRank {int(row['rank_test_score'])}:")
    print(f"  Score: {row['mean_test_score']:.4f} (+/- {row['std_test_score']:.4f})")
    print(f"  Params: {row['params']}")

# Demonstrate why pipeline is essential for grid search
print("\n" + "="*60)
print("WHY PIPELINE IS ESSENTIAL FOR GRIDSEARCH")
print("="*60)

print("\nWithout Pipeline:")
print("  Must manually preprocess for each parameter combination")
print("  Risk of data leakage in CV folds")
print("  Can't tune preprocessing hyperparameters")
print("  Code becomes complex and error-prone")

print("\nWith Pipeline:")
print("  Preprocessing automatically applied correctly")
print("  No leakage - each fold properly isolated")
print("  Can tune both preprocessing and model")
print("  Clean, maintainable code")
print("  Production-ready output")

##### **Rezultatul asteptat**
```
============================================================
PIPELINE HYPERPARAMETER TUNING
============================================================

Grid Search Completed!

Best Parameters:
  classifier__C: X.XX
  classifier__max_iter: XXX
  ...

Performance:
  Best CV Score: 0.9XXX
  Test Score:    0.9XXX

Top 5 Configurations:

Rank 1:
  Score: 0.9XXX (+/- 0.0XXX)
  Params: {...}
```
GridSearchCV ar trebui sa gaseasca hiperparametrii optimi, mentinand in acelasi timp izolarea corecta a datelor prin pipeline.


In [ ]:
unittests.exercise_4(tune_pipeline_hyperparameters)

<a name='6'></a>
## 6 - Pipeline de productie

Un **pipeline de productie** gestioneaza datele reale end-to-end: de la input brut pana la predictii, incluzand tratarea erorilor, validare si serializare.

### Cerinte de productie:

1. **Robustete**: gestioneaza valori lipsa, outlieri, categorii necunoscute
2. **Serializare**: salveaza si incarca intregul pipeline
3. **Validare**: verifica formatul si intervalele datelor de intrare
4. **Logging**: urmareste predictiile si potentialele probleme
5. **Versionare**: urmareste versiunea pipeline-ului si datele de antrenare
6. **Documentatie**: specificatii clare pentru input / output

### Pipeline complet de productie:

```python
# Numeric pipeline: imputation -> scaling
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())  # Robust to outliers
])

# Categorical pipeline: imputation -> encoding
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))  # Graceful handling
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Full pipeline
production_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
```

### Serializare:

**Salvare pipeline:**
```python
import joblib

# Save
joblib.dump(pipeline, 'model_pipeline.pkl')

# Load
loaded_pipeline = joblib.load('model_pipeline.pkl')

# Use immediately
predictions = loaded_pipeline.predict(new_data)
```

### Validarea input-ului:

```python
def validate_input(X, expected_columns):
    '''Validate input data before prediction.'''
    # Check column names
    if not all(col in X.columns for col in expected_columns):
        raise ValueError(f"Missing columns. Expected: {expected_columns}")
    
    # Check data types
    for col in numeric_columns:
        if not np.issubdtype(X[col].dtype, np.number):
            raise TypeError(f"{col} must be numeric")
    
    return X[expected_columns]  # Ensure column order
```

### Versionare si metadata:

```python
pipeline_metadata = {
    'version': '1.0.0',
    'trained_date': '2024-01-15',
    'training_samples': 10000,
    'features': list(X_train.columns),
    'target': 'outcome',
    'performance': {
        'train_accuracy': 0.95,
        'test_accuracy': 0.92
    }
}

# Save alongside pipeline
joblib.dump({
    'pipeline': pipeline,
    'metadata': pipeline_metadata
}, 'model_bundle.pkl')
```

### Tipare de deployment:

1. **Batch Prediction**:
   - incarci pipeline-ul o singura data
   - procesezi multe exemple
   - scrii rezultatele in baza de date / fisier

2. **Serviciu API**:
   - incarci pipeline-ul la pornire
   - gestionezi request-uri individuale
   - returnezi predictii in timp real

3. **Streaming**:
   - pipeline in procesorul de stream
   - procesezi inregistrari pe masura ce sosesc
   - predictii cu latenta mica


<a name='ex-5'></a>
### Exercitiul 5 - Pipeline de productie end-to-end

**Sarcina ta:**

Implementeaza functia `create_production_pipeline` astfel incat:
1. Sa construiasca un pipeline complet de preprocesare + model
2. Sa gestioneze elegant valorile lipsa si categoriile necunoscute
3. Sa antreneze pe date si sa evalueze performanta
4. Sa serializeze pipeline-ul pe disc
5. Sa incarce si sa valideze pipeline-ul salvat

<details>
  <summary><b><font color="green">Indicii de cod (apasati pentru extindere daca ai nevoie de ajutor)</font></b></summary>
  
**Pentru preprocesare robusta:**
* Numeric: SimpleImputer(strategy='median') + RobustScaler()
* Categorical: SimpleImputer(strategy='most_frequent') + OneHotEncoder(handle_unknown='ignore')

**Pentru serializare:**
```python
import joblib
joblib.dump(pipeline, filepath)
loaded = joblib.load(filepath)
```

**Pentru validare:**
* Incarca pipeline-ul
* Fa predictie pe un exemplu din test
* Verifica formatul output-ului
* Verifica daca performanta se potriveste

**Pentru metadata:**
* Include: data antrenarii, numele caracteristicilor, metrici de performanta
* Salveaza-le alaturi de pipeline

</details>


In [ ]:
# GRADED FUNCTION: create_production_pipeline
def create_production_pipeline(X_train, X_test, y_train, y_test, save_path='production_pipeline.pkl'):
    """
    Create, train, and serialize a production-ready pipeline.
    
    Args:
        X_train: Training features (DataFrame)
        X_test: Test features (DataFrame)
        y_train: Training labels
        y_test: Test labels
        save_path: Path to save serialized pipeline
    
    Returns:
        Dictionary with pipeline, scores, and validation results
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 35 lines)
    
    # Identify column types
    numeric_features = None
    categorical_features = None
    
    # Create numeric preprocessing pipeline
    numeric_transformer = None
    
    # Create categorical preprocessing pipeline
    categorical_transformer = None
    
    # Combine with ColumnTransformer
    preprocessor = None
    
    # Create full pipeline
    pipeline = None
    
    # Train pipeline
    # pipeline.fit(...)
    
    # Evaluate
    train_score = None
    test_score = None
    
    # Create metadata
    from datetime import datetime
    metadata = None  # Create dictionary with version, date, features, performance
    
    # Serialize pipeline and metadata
    bundle = {'pipeline': pipeline, 'metadata': metadata}
    # joblib.dump(...)
    
    # Validate serialization: Load and test
    loaded_bundle = None  # joblib.load(...)
    loaded_pipeline = None
    
    # Make test prediction to verify
    test_sample = None  # Use first test sample
    original_pred = None
    loaded_pred = None
    
    serialization_valid = None  # Check if predictions match
    
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'pipeline': pipeline,
        'train_score': train_score,
        'test_score': test_score,
        'metadata': metadata,
        'save_path': save_path,
        'serialization_valid': serialization_valid,
        'loaded_pipeline': loaded_pipeline
    }

In [ ]:
# Load mixed-type dataset
X, y = load_titanic_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("="*60)
print("PRODUCTION PIPELINE CREATION")
print("="*60)

results = create_production_pipeline(X_train, X_test, y_train, y_test)

print(f"\nPipeline created and trained!")
print(f"\nPerformance:")
print(f"  Training accuracy: {results['train_score']:.4f}")
print(f"  Test accuracy:     {results['test_score']:.4f}")

print(f"\nMetadata:")
for key, value in results['metadata'].items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    elif isinstance(value, list) and len(value) > 5:
        print(f"  {key}: {value[:5]}... ({len(value)} total)")
    else:
        print(f"  {key}: {value}")

print(f"\nSerialization:")
print(f"  Saved to: {results['save_path']}")
print(f"  Validation: {'PASSED' if results['serialization_valid'] else 'FAILED'}")

# Demonstrate production usage
print("\n" + "="*60)
print("PRODUCTION USAGE DEMONSTRATION")
print("="*60)

# Simulate loading in production
loaded_pipeline = results['loaded_pipeline']

# Make predictions on new data
sample_data = X_test.iloc[:3]
predictions = loaded_pipeline.predict(sample_data)
probabilities = loaded_pipeline.predict_proba(sample_data)

print(f"\nSample Predictions:")
for i, (idx, row) in enumerate(sample_data.iterrows()):
    print(f"\nSample {i+1}:")
    print(f"  Input: {dict(row)}")
    print(f"  Prediction: {'Survived' if predictions[i] == 1 else 'Did not survive'}")
    print(f"  Probability: {probabilities[i][1]:.2%}")

# Production checklist
print("\n" + "="*60)
print("PRODUCTION READINESS CHECKLIST")
print("="*60)

checklist = [
    ("Handles missing values", True),
    ("Handles unknown categories", True),
    ("Serializable", results['serialization_valid']),
    ("Includes metadata", results['metadata'] is not None),
    ("No data leakage", True),
    ("Version tracked", 'version' in results['metadata']),
    ("Performance documented", 'performance' in results['metadata'])
]

for item, status in checklist:
    symbol = "[x]" if status else "[ ]"
    print(f"{symbol} {item}")

print("\nThis pipeline is ready for:")
print("  • Batch predictions on new data")
print("  • REST API deployment")
print("  • Streaming applications")
print("  • Model registry storage")

##### **Rezultatul asteptat**
```
============================================================
CREAREA PIPELINE-ULUI DE PRODUCTIE
============================================================

Pipeline created and trained!

Performance:
  Training accuracy: 0.XXXX
  Test accuracy:     0.XXXX

Metadata:
  version: 1.0.0
  trained_date: 2024-XX-XX
  training_samples: XXX
  features: ['pclass', 'sex', 'age', ...]... (X total)
  performance:
    train_accuracy: 0.XXXX
    test_accuracy: 0.XXXX

Serialization:
  Saved to: production_pipeline.pkl
  Validation: PASSED

============================================================
CHECKLIST DE PREGATIRE PENTRU PRODUCTIE
============================================================
[x] Handles missing values
[x] Handles unknown categories
[x] Serializable
[x] Includes metadata
[x] No data leakage
[x] Version tracked
[x] Performance documented
```
Pipeline-ul de productie ar trebui sa gestioneze toate cazurile limita si sa poata fi serializat complet pentru deployment.


In [ ]:
unittests.exercise_5(create_production_pipeline)

<a name='7'></a>
## 7 - Implementare de la zero: pipeline simplu

Intelegerea modului in care functioneaza pipeline-urile in interior te ajuta sa depanezi probleme si sa creezi transformere personalizate.

### Logica de baza a pipeline-ului:

Un pipeline este, in esenta:
1. o lista de tupluri (nume, transformer / estimator)
2. logica pentru a lega apelurile fit / transform
3. tratare speciala pentru estimatorul final

### Metode cheie:

**fit(X, y):**
```python
def fit(self, X, y=None):
    X_transformed = X
    
    # Fit and transform all steps except last
    for name, transform in self.steps[:-1]:
        X_transformed = transform.fit_transform(X_transformed, y)
    
    # Fit final estimator
    self.steps[-1][1].fit(X_transformed, y)
    
    return self
```

**predict(X):**
```python
def predict(self, X):
    X_transformed = X
    
    # Transform through all steps except last (no fitting!)
    for name, transform in self.steps[:-1]:
        X_transformed = transform.transform(X_transformed)
    
    # Predict with final estimator
    return self.steps[-1][1].predict(X_transformed)
```

### Transform vs Fit_Transform:

- **fit_transform(X, y)**: invata din X si il transforma
- **transform(X)**: aplica transformarea invatata pe X

In training: folosesti `fit_transform` pentru eficienta

La predictie: doar `transform` (fara fitting!)

### Transformer personalizat:

```python
from sklearn.base import BaseEstimator, TransformerMixin

class CustomTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, param=1.0):
        self.param = param
    
    def fit(self, X, y=None):
        # Learn something from X
        self.learned_value_ = X.mean()
        return self
    
    def transform(self, X):
        # Apply transformation
        return X * self.param
```


<a name='ex-6'></a>
### Exercitiul 6 - Pipeline simplu de la zero

**Sarcina ta:**

Implementeaza clasa `SimplePipeline` astfel incat:
1. Sa stocheze o lista de transformere si un estimator final
2. Sa implementeze `fit()` pentru a lega operatiile fit/transform
3. Sa implementeze `predict()` pentru a transforma si prezice
4. Sa valideze ca implementarea ta se potriveste cu Pipeline-ul din sklearn

<details>
  <summary><b><font color="green">Indicii de cod (apasati pentru extindere daca ai nevoie de ajutor)</font></b></summary>
  
**Pentru initializare:**
* Stocheaza pasii ca lista de tuple (nume, transformer)
* Verifica faptul ca toate au fit/transform (cu exceptia ultimului)

**Pentru fit():**
* Itereaza prin toti pasii, cu exceptia ultimului
* Pentru fiecare: apeleaza fit_transform() si paseaza rezultatul mai departe
* Pentru ultimul pas: apeleaza doar fit()
* Returneaza self pentru chaining

**Pentru predict():**
* Itereaza prin toti pasii, cu exceptia ultimului
* Pentru fiecare: apeleaza doar transform() (fara fitting!)
* Pentru ultimul pas: apeleaza predict()

**Pentru score():**
* Transforma X prin toti pasii, cu exceptia ultimului
* Apeleaza score() pe estimatorul final

</details>


In [ ]:
# GRADED FUNCTION: SimplePipeline
class SimplePipeline:
    """
    Simple pipeline implementation from scratch.
    
    Chains transformers and a final estimator, ensuring proper
    fit/transform order to prevent data leakage.
    """
    
    def __init__(self, steps):
        """
        Initialize pipeline with list of (name, transformer) tuples.
        
        Args:
            steps: List of (name, transformer) tuples
        """
        ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
        self.steps = None
        ### SFÂRȘIT DE COD AICI ###
    
    def fit(self, X, y=None):
        """
        Fit all transformers and final estimator.
        
        For each transformer: fit and transform
        For final estimator: only fit
        """
        ### ÎNCEPUT DE COD AICI ### (≈ 10 lines)
        
        X_transformed = None  # Start with original X
        
        # Fit and transform all steps except last
        for name, transformer in None:  # Loop through steps[:-1]
            # Fit and transform
            X_transformed = None
        
        # Fit final estimator (no transform)
        final_name, final_estimator = None
        # final_estimator.fit(...)
        
        ### SFÂRȘIT DE COD AICI ###
        
        return self
    
    def predict(self, X):
        """
        Transform X through all transformers, then predict with final estimator.
        
        Important: Only transform, never fit during prediction!
        """
        ### ÎNCEPUT DE COD AICI ### (≈ 8 lines)
        
        X_transformed = None  # Start with original X
        
        # Transform through all steps except last (NO FITTING!)
        for name, transformer in None:  # Loop through steps[:-1]
            X_transformed = None  # Only transform
        
        # Predict with final estimator
        final_name, final_estimator = None
        predictions = None
        
        ### SFÂRȘIT DE COD AICI ###
        
        return predictions
    
    def score(self, X, y):
        """
        Transform X and score with final estimator.
        """
        ### ÎNCEPUT DE COD AICI ### (≈ 8 lines)
        
        X_transformed = None
        
        # Transform through all steps except last
        for name, transformer in None:
            X_transformed = None
        
        # Score with final estimator
        final_name, final_estimator = None
        score = None
        
        ### SFÂRȘIT DE COD AICI ###
        
        return score

In [ ]:
# Test custom pipeline implementation
print("="*60)
print("SIMPLE PIPELINE FROM SCRATCH")
print("="*60)

# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create sklearn pipeline for comparison
sklearn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Create custom pipeline
custom_pipeline = SimplePipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Train both
print("\nTraining pipelines...")
sklearn_pipeline.fit(X_train, y_train)
custom_pipeline.fit(X_train, y_train)

# Compare predictions
sklearn_pred = sklearn_pipeline.predict(X_test)
custom_pred = custom_pipeline.predict(X_test)

predictions_match = np.array_equal(sklearn_pred, custom_pred)

print(f"\nPredictions match: {predictions_match}")

# Compare scores
sklearn_score = sklearn_pipeline.score(X_test, y_test)
custom_score = custom_pipeline.score(X_test, y_test)

print(f"\nPerformance Comparison:")
print(f"  Sklearn Pipeline: {sklearn_score:.4f}")
print(f"  Custom Pipeline:  {custom_score:.4f}")
print(f"  Difference:       {abs(sklearn_score - custom_score):.6f}")

if abs(sklearn_score - custom_score) < 1e-6:
    print("\nYour pipeline implementation is correct!")
else:
    print("\nScores don't match. Check your implementation.")

# Demonstrate that pipeline prevents leakage
print("\n" + "="*60)
print("LEAKAGE PREVENTION DEMONSTRATION")
print("="*60)

# Check that scaler was fit only on training data
scaler = custom_pipeline.steps[0][1]
print(f"\nScaler learned from training data:")
print(f"  Mean: {scaler.mean_[:3]}...")
print(f"  Std:  {scaler.scale_[:3]}...")

# Compare with scaler fit on all data (WRONG!)
scaler_leaky = StandardScaler().fit(np.vstack([X_train, X_test]))
print(f"\nLeaky scaler (fit on all data):")
print(f"  Mean: {scaler_leaky.mean_[:3]}...")
print(f"  Std:  {scaler_leaky.scale_[:3]}...")

print(f"\nThe values are different because our pipeline correctly")
print(f"   fits the scaler only on training data, preventing leakage!")

# Show fit/transform chaining
print("\n" + "="*60)
print("UNDERSTANDING FIT/TRANSFORM CHAINING")
print("="*60)

print("\nDuring Training (fit):")
print("  1. Scaler.fit_transform(X_train)  → X_scaled")
print("  2. Classifier.fit(X_scaled, y_train)")
print("  Scaler learns from training data only")

print("\nDuring Prediction (predict):")
print("  1. Scaler.transform(X_test)  → X_scaled (NO FITTING!)")
print("  2. Classifier.predict(X_scaled)")
print("  No leakage: test data never influences training")

print("\nKey Insight: Pipeline ensures fit happens only during training,")
print("   and only transform is used during prediction. This is the core")
print("   mechanism that prevents data leakage!")

##### **Rezultatul asteptat**
```
============================================================
SIMPLE PIPELINE FROM SCRATCH
============================================================

Training pipelines...

Predictions match: True

Performance Comparison:
  Sklearn Pipeline: 0.9XXX
  Custom Pipeline:  0.9XXX
  Difference:       0.000000

Your pipeline implementation is correct!

============================================================
DEMONSTRATIE DE PREVENIRE A LEAKAGE-ULUI
============================================================

Scaler learned from training data:
  Mean: [X.XX X.XX X.XX]...
  Std:  [X.XX X.XX X.XX]...

Leaky scaler (fit on all data):
  Mean: [X.XX X.XX X.XX]...
  Std:  [X.XX X.XX X.XX]...
```
Pipeline-ul tau personalizat ar trebui sa produca aceleasi rezultate ca Pipeline-ul din sklearn, demonstrand o implementare corecta.


In [ ]:
unittests.exercise_6(SimplePipeline)

<a name='8'></a>
## 8 - Concluzie

**Felicitari!** Ai finalizat tema despre Pipeline-uri ML si Prevenirea Data Leakage!

### Ce ai invatat:

1. **Pipeline-uri de baza**: leaga preprocesarea si modelarea cu ordinea corecta fit/transform
2. **ColumnTransformer**: gestioneaza elegant date numerice si categorice mixte
3. **Detectarea leakage-ului**: identifica tipare comune de leakage si le corecteaza
4. **Pipeline Tuning**: integreaza sigur cautarea de hiperparametri cu pipeline-uri
5. **Pipeline-uri de productie**: construieste sisteme robuste, serializabile, end-to-end
6. **Implementare de la zero**: intelege mecanica interna a pipeline-urilor

### Idei principale:

**Foloseste mereu pipeline-uri:**
- previn automat data leakage
- fac codul mai curat si mai usor de intretinut
- asigura consistenta intre training si predictie
- simplifica deployment-ul si serializarea
- functioneaza natural cu cross-validation

**Prevenirea data leakage:**
- fa split-ul datelor INAINTE de orice preprocesare
- fa fit pe transformere doar pe datele de antrenare
- foloseste pipeline-uri pentru toata preprocesarea
- respecta ordinea temporala in serii de timp
- valideaza pe date cu adevarat hold-out

**Bune practici pentru productie:**
- gestioneaza elegant valori lipsa si outlieri
- foloseste `handle_unknown='ignore'` in encodere
- include metadata (versiune, data, performanta)
- testeaza serializarea si deserializarea
- valideaza datele de intrare inainte de predictie
- logheaza predictiile si monitorizeaza performanta

### Greseli comune de evitat:

**Preprocesare inainte de split**
```python
# WRONG
X_scaled = scaler.fit_transform(X)
X_train, X_test = train_test_split(X_scaled)
```

**Corect: fa split mai intai**
```python
# CORRECT
X_train, X_test = train_test_split(X)
pipeline = make_pipeline(scaler, model)
pipeline.fit(X_train, y_train)
```

**Preprocesare in afara CV**
```python
# WRONG
X_scaled = scaler.fit_transform(X)
cross_val_score(model, X_scaled, y)
```

**Corect: pipeline in CV**
```python
# CORRECT
pipeline = make_pipeline(scaler, model)
cross_val_score(pipeline, X, y)
```

**Folosirea unor caracteristici care scurg tinta**
```python
# WRONG: Feature contains target information
features = ['income', 'already_defaulted']  # This is the target!
```

**Corect: doar caracteristici disponibile inainte de tinta**
```python
# CORRECT: Only features available before target
features = ['income', 'credit_score', 'employment_years']
```

### Aplicatii in lumea reala:

**Servicii financiare:**
- credit scoring: previi leakage-ul temporal
- detectie de frauda: servire in timp real cu pipeline-uri
- risk modeling: gestionare robusta a datelor lipsa

**Healthcare:**
- predictia evolutiei pacientului: fara informatie din viitor
- diagnostic medical: gestionezi inregistrari incomplete
- eficacitate tratamente: split-uri clinice corecte

**E-commerce:**
- churn: validare temporala
- sisteme de recomandare: split bazat pe utilizator
- prognoza cererii: pipeline-uri pentru serii de timp

**Manufacturing:**
- quality control: predictii in timp real
- predictive maintenance: preprocesare pentru date de senzori
- supply chain: date multi-modale

### Subiecte avansate:

**Transformere personalizate:**
- mostenesc din `BaseEstimator` si `TransformerMixin`
- implementeaza `fit()` si `transform()`
- adauga logica de preprocesare personalizata

**Feature Unions:**
- cai paralele de extragere a caracteristicilor
- combina multiple seturi de caracteristici
- utile pentru date eterogene

**Pipeline Caching:**
- parametrul `memory` pentru transformari costisitoare
- accelereaza hyperparameter tuning
- util pentru feature engineering

**Model Stacking:**
- pipeline-uri ca estimatori de baza
- meta-learner care combina predictiile
- tehnici avansate de ensemble

### Instrumente si bune practici:

**Development:**
- foloseste pipeline-uri de la inceput
- testeaza cu date de exemplu
- valideaza serializarea devreme
- documenteaza cerintele privind caracteristicile

**Productie:**
- tine pipeline-urile sub version control
- monitorizeaza distributiile de intrare
- urmareste performanta predictiilor
- fa A/B testing pentru versiunile noi

**Debugging:**
- foloseste `set_config(display='diagram')`
- inspecteaza output-urile intermediare
- verifica parametrii invatati
- valideaza fata de abordarea manuala

### Resurse recomandate:

**Documentatie:**
- Scikit-learn Pipeline User Guide
- exemplele Scikit-learn pentru Column Transformer
- ghiduri de bune practici MLOps

**Carti:**
- "Building Machine Learning Powered Applications" (Ameisen)
- "Designing Machine Learning Systems" (Huyen)
- "Machine Learning Design Patterns" (Lakshmanan et al.)

**Articole:**
- "Leakage in Data Mining" (Kaufman et al.)
- "Rules of Machine Learning" (Google)
- "Hidden Technical Debt in ML Systems" (Sculley et al.)

### Checklist final pentru productie:

Inainte sa deploy-ezi pipeline-ul:

- [ ] Nu exista data leakage (verificat cu CV corect)
- [ ] Gestioneaza valori lipsa
- [ ] Gestioneaza categorii necunoscute
- [ ] Exista validare pentru input
- [ ] Serializarea este testata
- [ ] Metadata sunt incluse
- [ ] Performanta este documentata
- [ ] Versiunea este urmarita
- [ ] Exista un plan de monitorizare
- [ ] Exista o strategie de rollback

**Foarte bine! Acum esti pregatit sa construiesti pipeline-uri ML gata de productie, care previn data leakage si pot fi deploy-ate in siguranta!**
